# 图像分割 · 标准答案

这份完整程序与题目版使用相同数据和流程，可以运行并对照各处代码。训练结果取决于设置与运行条件。

第一次按顺序运行数据检查和图像显示，再补全各模块设置。题目版初始 `RUN_TRAINING=False`（本参考版开启训练），会显示尚未启用训练的提示；模块完成后改为 True，从准备格起按顺序重跑。它会重新创建两种方案并训练，不是接着上一轮追加训练。

In [ ]:
from pathlib import Path
import os, json, random, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('project_outputs')
WORK.mkdir(exist_ok=True)
SIZE, BATCH, EPOCHS, LR = 128, 32, 5, 0.001  # 修改尺寸或轮数后，重新运行后续单元格
print('device:', DEVICE, 'output:', WORK)
def show(images, titles, path=None):
    fig, axes = plt.subplots(1,len(images),figsize=(4*len(images),4),squeeze=False)
    for ax, im, title in zip(axes[0],images,titles):
        if isinstance(im,torch.Tensor): im=im.detach().cpu().permute(1,2,0).numpy()
        ax.imshow(im); ax.set_title(title); ax.axis('off')
    fig.tight_layout()
    if path: fig.savefig(path,dpi=150,bbox_inches='tight')
    plt.show(); plt.close(fig)
def ready(*items):
    ok = all(item is not Ellipsis for item in items)
    if not ok: print('本部分仍有 ... 待填；补全后重新运行本格及后续代码。')
    return ok

RUN_TRAINING = True  # 补全代码后改为 True；训练 A、B 两个方案

## 数据与划分

Input 中已添加完整 Oxford-IIIT Pet 数据。代码读取官方 `trainval.txt` 和 `test.txt`，共 7349 张有官方划分的照片；其余未进入这两份清单的图片不加入实验。官方测试集保留不变，trainval 按猫狗比例划出 20% 验证集。

来源：Parkhi et al., Cats and Dogs, CVPR 2012；Oxford-IIIT Pet，CC BY-SA 4.0。

`SIZE` 是输入图片边长，`BATCH` 是每批图片数，`EPOCHS` 是完整遍历训练集的轮数，`LR` 是学习率，`SEED` 固定随机起点。保持默认值完成首次运行。数据已随 Kaggle 项目挂载；本地下载运行才需准备对应数据目录。

In [ ]:
INPUT = Path(os.environ.get('KYDW_PET_ROOT','/kaggle/input'))
roots=[INPUT] if (INPUT/'annotations/trainval.txt').exists() else [p.parent.parent for p in INPUT.rglob('trainval.txt') if p.parent.name=='annotations']
assert roots, '请在 Add Input 中添加 the-oxfordiiit-pet-dataset，包含 images 和 annotations。'
ROOT=roots[0]
def read_split(name):
    rows=[]
    for line in (ROOT/'annotations'/f'{name}.txt').read_text().splitlines():
        if not line.strip() or line.startswith('#'): continue
        image,breed,species,*_=line.split()
        rows.append(dict(id=image,label=int(species)-1,breed=int(breed),image=str(ROOT/'images'/f'{image}.jpg'),mask=str(ROOT/'annotations/trimaps'/f'{image}.png')))
    return pd.DataFrame(rows)
trainval, test = read_split('trainval'), read_split('test')
train, val = train_test_split(trainval,test_size=0.2,random_state=SEED,stratify=trainval.label)
train,val,test=[d.reset_index(drop=True) for d in [train,val,test]]
assert not (set(train.id)&set(val.id) or set(trainval.id)&set(test.id))
all_rows=pd.concat([train.assign(split='train'),val.assign(split='val'),test.assign(split='test')])
assert all(Path(p).is_file() for p in all_rows.image), '存在缺失照片'
assert all(Path(p).is_file() for p in all_rows['mask']), '存在缺失分割标注'
all_rows[['id','label','breed','split']].to_csv(WORK/'split.csv',index=False)
print(all_rows.groupby(['split','label']).size().unstack(fill_value=0).rename(columns={0:'cat',1:'dog'}))
print('官方 trainval:',len(trainval),'官方 test:',len(test),'实际使用总数:',len(all_rows))

## 照片、标签与尺寸分布

同一张照片对应整图猫狗标签和逐像素 trimap。修改 `SAMPLE_INDEX` 可以查看另一只动物；下面的直方图统计训练照片的原始尺寸。

In [ ]:
SAMPLE_INDEX = 0  # 改成另一行编号，观察照片、原始尺寸与标签
row=train.iloc[SAMPLE_INDEX]
photo=Image.open(row.image).convert('RGB'); mask=Image.open(row['mask'])
print('id:',row.id,'size:',photo.size,'species:',['cat','dog'][row.label])
show([photo,np.asarray(mask)],['Original photo','Trimap: 1 pet / 2 background / 3 boundary'])
sizes=np.array([Image.open(p).size for p in train.image])
fig,axes=plt.subplots(1,2,figsize=(10,3))
axes[0].hist(sizes[:,0],bins=30,label='width',alpha=.6);axes[0].hist(sizes[:,1],bins=30,label='height',alpha=.6);axes[0].legend();axes[0].set_xlabel('Original pixels')
train.label.value_counts().sort_index().plot.bar(ax=axes[1]);axes[1].set_xticklabels(['cat','dog'],rotation=0);axes[1].set_ylabel('Training images');plt.tight_layout();plt.show();plt.close(fig)

## 前景与待定边界

官方 trimap 使用 1 表示宠物、2 表示背景、3 表示待定边界。填写前景编码和忽略编码。照片与标注共享翻转；标注缩放使用最近邻，避免产生不存在的类别。

`FOREGROUND` 和 `IGNORE` 对应 trimap 像素编码，和整图猫狗 species 不是同一种标签。模型仍预测忽略区，只是损失与指标不统计这些位置。

In [ ]:
FOREGROUND = 1  # 对应：宠物编码
IGNORE = 3  # 对应：不参与损失与评价的编码
class PetDataset(Dataset):
    def __init__(self,rows,flip=0):self.rows=rows;self.flip=flip
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i]
        im=Image.open(row.image).convert('RGB').resize((SIZE,SIZE),Image.Resampling.BILINEAR)
        mask=Image.open(row['mask']).resize((SIZE,SIZE),Image.Resampling.NEAREST)
        if random.random()<self.flip:
            im=im.transpose(Image.Transpose.FLIP_LEFT_RIGHT);mask=mask.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        x=torch.from_numpy(np.asarray(im).copy()).permute(2,0,1).float()/255
        m=torch.from_numpy(np.asarray(mask).copy())
        return x,(m==FOREGROUND).float()[None],(m!=IGNORE)[None],row.id
if ready(FOREGROUND,IGNORE):
    assert FOREGROUND in [1,2,3] and IGNORE in [1,2,3] and FOREGROUND!=IGNORE
    x,y,valid,_=PetDataset(train)[0]
    show([x,y[0].numpy(),valid[0].numpy()],['Input','Foreground target','Valid pixels'])

## 网络与概率阈值

每个像素只需一个前景 logit，Sigmoid 将它转换为前景概率。填写输出通道数，并选择 0.5 作为初始分割阈值。U-Net 的编码、跳跃连接和解码已经提供。

网络中的 `BatchNorm2d` 在训练时按批次调整中间特征的数值尺度，预测时使用训练中保存的统计量；`train()` 与 `eval()` 分别控制两种状态。

In [ ]:
OUT_CHANNELS = 1  # 对应：前景 logit 通道数
THRESHOLD = 0.5  # 对应：概率阈值
class SmallUNet(nn.Module):
    def __init__(self,out_channels):
        super().__init__()
        def block(a,b): return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(),nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU())
        self.e1=block(3,8);self.e2=block(8,16);self.mid=block(16,32)
        self.d2=block(48,16);self.d1=block(24,8);self.out=nn.Conv2d(8,out_channels,1)
    def forward(self,x):
        a=self.e1(x);b=self.e2(F.max_pool2d(a,2));c=self.mid(F.max_pool2d(b,2))
        d=self.d2(torch.cat([F.interpolate(c,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1))
        e=self.d1(torch.cat([F.interpolate(d,size=a.shape[-2:],mode='bilinear',align_corners=False),a],1))
        return self.out(e)

## 忽略区与损失

逐像素二元交叉熵保留每个位置的损失，再用有效掩膜筛选求平均。填写损失的 `reduction`，让输出仍保留空间位置；只要默认求均值，就无法再排除待定区。

In [ ]:
REDUCTION = 'none'  # 对应：不汇总损失时的 reduction 字符串

## 训练两个方案

A、B 使用相同网络、种子和划分，仅水平翻转概率不同。验证损失决定保存哪一轮权重；损失将有效区的像素交叉熵与前景 soft Dice 损失相加，既比较每个位置，也比较前景重合；待定边界不参与两项计算。补全后把 `RUN_TRAINING` 改为 True。

运行时每轮显示方案名、轮次、训练损失与验证损失。正在运行的格会显示忙碌标记；训练格尚未结束时，不运行评价格。显存不足可减小 `BATCH` 后从准备格重跑，A/B 都采用相同的新设置。

In [ ]:
def epoch(model,loader,optimizer=None):
    model.train(optimizer is not None);total,count=0.,0
    with torch.set_grad_enabled(optimizer is not None):
        for batch in loader:
            x,target=batch[0].to(DEVICE),batch[1].to(DEVICE)
            pred=model(x)
            assert pred.shape==target.shape, "预测与参考形状不一致，请核对输出通道和空间尺寸"
            assert torch.isfinite(pred).all(), "预测出现非有限数值，请检查输入、损失和学习率"
            valid=batch[2].to(DEVICE)
            pixel=F.binary_cross_entropy_with_logits(pred,target,reduction=REDUCTION)
            pixel_loss=(pixel*valid).sum()/valid.sum().clamp_min(1)
            probability=torch.sigmoid(pred)*valid
            truth=target*valid
            overlap=(probability*truth).sum((1,2,3))
            dice=(2*overlap+1e-6)/(probability.sum((1,2,3))+truth.sum((1,2,3))+1e-6)
            loss=pixel_loss+(1-dice).mean()  # 像素误差与前景重合共同参与更新
            if optimizer is not None:
                optimizer.zero_grad();loss.backward();optimizer.step()
            total+=loss.item()*len(x);count+=len(x)
    return total/count

def run_experiment(name,flip):
    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
    model=SmallUNet(OUT_CHANNELS).to(DEVICE)
    optimizer=torch.optim.Adam(model.parameters(),lr=LR)
    train_loader=DataLoader(PetDataset(train,flip),batch_size=BATCH,shuffle=True,num_workers=0)
    val_loader=DataLoader(PetDataset(val),batch_size=BATCH,num_workers=0)
    history=[];best=float('inf');folder=WORK/name;folder.mkdir(exist_ok=True)
    for step in range(EPOCHS):
        a=epoch(model,train_loader,optimizer);b=epoch(model,val_loader)
        history.append(dict(epoch=step+1,train_loss=a,val_loss=b));print(name,history[-1])
        if b<best:
            best=b;torch.save(model.state_dict(),folder/'best.pt')
    pd.DataFrame(history).to_csv(folder/'history.csv',index=False)
    (folder/'config.json').write_text(json.dumps(dict(seed=SEED,size=SIZE,batch=BATCH,epochs=EPOCHS,lr=LR,flip=flip,train=len(train),val=len(val),test=len(test)),indent=2))
    model.load_state_dict(torch.load(folder/'best.pt',map_location=DEVICE,weights_only=True));model.eval()
    return model,folder
models={}
if RUN_TRAINING and ready(FOREGROUND,IGNORE,OUT_CHANNELS,THRESHOLD,REDUCTION):
    for name,flip in [('A',0.0),('B',0.5)]: models[name]=run_experiment(name,flip)
else: print('补全待填处后，将 RUN_TRAINING 改为 True，再运行本格。A 与 B 只改变水平翻转概率。')

## 逐图评价与对照图

Dice、IoU 对每张图的有效像素计算后取平均。真实与预测都没有前景时记为 1；完全没有有效像素的图片记录为缺失，不计入平均。固定展示测试清单前三张图，并保存逐图差值。

模型 logits 经 Sigmoid 得到概率，阈值转成 0/1 后计算 Dice 指标；训练用的 soft Dice 则保留连续概率。输出通道不等于类别编号。

In [ ]:
def per_image_scores(pred,target,valid):
    p=pred.bool()&valid;t=target.bool()&valid
    dims=(1,2,3);inter=(p&t).sum(dims);den=p.sum(dims)+t.sum(dims);union=(p|t).sum(dims)
    dice=torch.where(den>0,2*inter/den,torch.ones_like(den,dtype=torch.float32))
    iou=torch.where(union>0,inter/union,torch.ones_like(union,dtype=torch.float32))
    has_valid=valid.sum(dims)>0
    return dice,iou,has_valid
for name,(model,folder) in models.items():
    rows=[]
    with torch.no_grad():
        for x,y,valid,ids in DataLoader(PetDataset(test),batch_size=BATCH):
            pred=torch.sigmoid(model(x.to(DEVICE))).cpu()>=THRESHOLD
            dice,iou,usable=per_image_scores(pred,y,valid)
            rows.extend(dict(id=i,dice=float(d) if ok else np.nan,iou=float(j) if ok else np.nan,valid=bool(ok)) for i,d,j,ok in zip(ids,dice,iou,usable))
    table=pd.DataFrame(rows);table.to_csv(folder/'metrics.csv',index=False)
    print(name,'test n:',len(table),'valid images:',int(table.valid.sum()),table[['dice','iou']].mean().to_dict())
    hist=pd.read_csv(folder/'history.csv');hist.plot(x='epoch',y=['train_loss','val_loss']);plt.savefig(folder/'curves.png',dpi=150);plt.show();plt.close()
    for i in range(3):
        x,y,valid,identifier=PetDataset(test)[i]
        with torch.no_grad():pred=(torch.sigmoid(model(x[None].to(DEVICE)))[0,0].cpu()>=THRESHOLD).numpy()
        show([x,y[0].numpy(),pred,valid[0].numpy()],['Input','Target','Prediction','Valid pixels'],folder/f'{identifier}.png')
if len(models)==2:
    a=pd.read_csv(WORK/'A/metrics.csv');b=pd.read_csv(WORK/'B/metrics.csv');cmp=a.merge(b,on='id',suffixes=('_A','_B'),validate='one_to_one');cmp['dice_delta']=cmp.dice_B-cmp.dice_A;cmp.to_csv(WORK/'comparison.csv',index=False)
    cmp.dice_delta.hist(bins=30);plt.xlabel('Dice B - A');plt.savefig(WORK/'comparison.png',dpi=150);plt.show();plt.close()

## Word · 图像分割报告

最终只交一份 2—3 页报告，说明任务与数据、方法、结果、讨论和出处。使用自己的 A/B 实测指标和同样本配图；网页标准参考里的平移结果仅用于演示排版，不混入模型结果。

A4，四周页边距 2.5 cm；正文微软雅黑 11 pt、1.25 倍行距、段后 6 pt；题目 20 pt，一级标题 14 pt，题注 10 pt。使用标题样式、原生表格、自动题注、至少一处交叉引用与页码。A/B 两行表列出图像数、Dice 与 IoU，保留 3 位小数；至少两个样本按输入、参考、A、B 排列，注明忽略区和实际条件。

保存 `姓名_图像分割_报告.docx` 和同名 PDF，放入 `02_图像分割`。

## 保存项目

Notebook 的 Output / Files 区可查看 `/kaggle/working`。A、B 文件夹保存权重、设置、曲线与逐图结果；根目录保存固定划分和同编号的比较。训练结束后使用 **Save Version → Save & Run All** 保存可重复运行的版本，再下载修改后的 `.ipynb` 与主要结果；重跑全篇会重新训练。

代码、主要结果和相应 PPT、报告或海报放进同一个项目文件夹，再将三个文件夹打包为 `KYDW_姓名_培训考核.zip`。无需另交过程日志或自查表。